In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated
from pymongo import MongoClient
from datetime import datetime


In [4]:
client = MongoClient("mongodb://localhost:27017/")
db = client["miniproject"]

In [3]:
def create_thread(user_id : str, thread_number: int, thread_name : str) -> str:
    thread_id = f"{user_id}:{thread_number}"
    name = thread_name or f"Chat {thread_number}"

    now = datetime.now()

    existing = db.conversations.find_one({"thread_id" : thread_id})
    if existing:
        return thread_id
    
    db.conversations.insert_one({
        "user_id" : user_id,
        "thread_id" : thread_id,
        "thread_name" : name,
        "messages" : [],
        "created_at" : now,
        "updated_at" : now
    })
    return thread_id



def append_message(thread_id : str, role : str, content : str):
    message = {
        "role" : role,   # human or ai (later tool)
        "content" : content,
        "timestamp" : datetime.now()
    }

    db.conversations.update_one(
        {"thread_id" : thread_id}, {
            "$push" : {"messages" : message},
            "$set" : {"updated_at" : datetime.now()}
        }
    )


def load_thread ( thread_id : str) :
    doc = db.conversations.find_one({"thread_id" : thread_id})
    if not doc:
        return []
    return doc["messages"]


def load_thread_as_langchain_messages(thread_id: str):
    raw = load_thread(thread_id=thread_id)

    messages = []
    for m in raw:
        if m['role'] == "human":
            messages.append(HumanMessage(content=m['content']))
        elif m['role'] == 'ai':
            messages.append(AIMessage(content=m['content']))
    return messages



def get_user_threads(user_id : str):
    docs = db.conversations.find(
        {"user_id" : user_id},
        {"thread_id" : 1, "thread_name" : 1, "created_at" : 1, "updated_at" : 1, "messages" : 0}
    )

    threads = []
    for doc in docs:
        threads.append({
            "thread_id" : doc["thread_id"],
            "thread_name" : doc["thread_name"],
            "created_at" : doc["created_at"],
            "updated_at" : doc["updated_at"],
        })
    return threads


def get_next_thread_number(user_id : str):
    count = db.conversations.count_documents({"user_id" : user_id})
    return count + 1

def thread_exists(thread_id : str):
    return db.conversations.count_documents({"thread_id" : thread_id}) > 0


def load_state(thread_id : str):
    state = db.thread_state.find_one({"thread_id" : thread_id})
    if not state:
        return {
            "user_id" : "",
            "thread_id" : thread_id,
            "user_name" : "",
            "prakriti" : "",
            "bullet_points" : [],
            "is_risky" : False,
            "appointment_booked" : False
        }
    return state["state"]

In [10]:
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="fdadf",
    model="local-model"
)

In [11]:
class BotState(TypedDict):
    user_id : str
    thread_id : str
    user_name : str
    prakriti : str
    is_risky : bool
    appointment_booked : bool
    messages : Annotated[list, add_messages]


In [ ]:
system_prompt = "You are ayurbot"

In [ ]:
def call_llm(state: BotState):
    user  = db.users.find_one({"user_id" : state['user_id']} )
    user_name = user["fullName"] if user else ""

    prakriti  = user["prakriti"] if user else ""

    all_messages = [SystemMessage(content=f"{system_prompt}  \n User Name : {user_name}, Prakriti : {prakriti}"), *state['messages']]

    response = llm.invoke(all_messages)
    return {"user_name" : user_name, "messages" : [response]}

def save_state( state : BotState):
    # crete new document if not exists, else update
    thread_id =state['thread_id']
    db.thread_state.update_one(
        {"thread_id" : thread_id},
        {"$set" : {
            "user_id" : state['user_id'],
            "prakriti" : state['prakriti'],
            "is_risky" : state['is_risky'],
            "appointment_booked" : state['appointment_booked'],
            # "messages" : state['messages']
        }},
        upsert=True
    )

    return state

In [ ]:
memory = MemorySaver()

builder = StateGraph(BotState)
builder.add_node("call_llm", call_llm)
builder.add_node("save_state", save_state)

builder.set_entry_point("call_llm")

builder.add_edge("call_llm", "save_state")
builder.add_edge("save_state", END)

graph = builder.compile(checkpointer=memory)

In [ ]:
def run_chat(user_id : str, thread_id : str):
    config = {"configurable" : {"thread_id" : thread_id}}

    old_messages = load_thread_as_langchain_messages(thread_id)

    print(f"\n ------THread : {thread_id} | {"resuming" if old_messages else "New Chat"}---- \n")


    while True:
        user_input = input("You : ")
        if user_input.lower() in ["exit", "quit"]:
            print("Exiting chat")
            break

        append_message(thread_id, "human", user_input)

        # if thread_exists(thread_id):
        #     print("Thread exists, loading messages")
        # else:
        #     print("Thread does not exist, creating new thread")
        #     create_thread(user_id, get_next_thread_number(user_id), thread_name=None)

        messages = load_thread_as_langchain_messages(thread_id)
        state = load_state